[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/36.%20The%20Berth%20%26%20Quay%20Length%20Design%20Problem/P36-Tier-5_executed.ipynb)

---

# 36. The Berth & Quay Length Design Problem

## Tier 5: Integrated Digital-Twin Simulation

### Goal

This notebook builds an illustrative, integrated digital-twin simulation for a port. It connects a physical-asset model, operational data, predictive-analytics components, a time-stepped simulation, and decision-support tools. The simulation demonstrates how these layers can exchange information; it is not a live connection to port sensors or production systems.

### Inputs and outputs

The example uses a quay configuration, synthetic vessel traffic, synthetic sensor readings, and stated illustrative model assumptions. It produces simulated berth allocations, operational indicators, prediction records, alerts, visual summaries, and a comparison of infrastructure scenarios.

### Model assumptions and scope

- The example port has 3,200 m of quay divided into eight berths. The specified berth lengths sum to 3,200 m.
- The sensor readings, vessel traffic, model-accuracy figures, processing rate, and financial scenario assumptions are illustrative inputs. No trained prediction model, live sensor feed, verified forecast accuracy, or measured return on investment is supplied or evaluated here.
- Vessel assignment uses a transparent greedy rule: consider vessels in their current list order and assign each to the available berth with the lowest utilization rate among berths that meet its length and depth requirements.
- The time-stepped simulation advances in 30-second increments. This makes the example computationally practical while retaining the stated 30-second sensor-update interval and five-minute optimization interval.
- The simplified model omits detailed navigation, berth-conflict geometry, cargo-handling schedules, yard capacity constraints, uncertainty calibration, operating costs, taxes, and project financing. The integration classes illustrate interfaces and data flow; they do not constitute validated operational subsystems.

### Digital-twin architecture

The five layers have distinct roles: the **Physical Asset Layer** describes berths and sensors; the **Operational Data Layer** stores vessels and incoming operational records; the **Analytics Layer** produces illustrative predictions; the **Simulation Layer** advances the modeled port state; and the **Decision Support Layer** allocates berths and reports key performance indicators (KPIs).

A digital twin is useful here because a port decision can depend on multiple connected subsystems. The simulation makes these dependencies explicit, while its assumptions make clear which results are model outputs and which would require real data for operational use.

### Allocation rule

For vessel \(v\) and berth \(b\), the eligibility conditions are

$$
L_b \geq L_v, \qquad D_b \geq d_v, \qquad b \text{ is unoccupied and operational},
$$

where \(L_b\) and \(D_b\) are berth length and depth, and \(L_v\) and \(d_v\) are vessel length and draft. Among the eligible berths, the rule selects one with the smallest utilization rate. Vessels are processed sequentially, so an earlier assignment can affect later choices. This is a greedy allocation heuristic, not a proof of global optimality; the optimization-engine class implements this rule.

### Scenario-analysis calculation

For an infrastructure scenario, the illustrative financial calculation treats the scenario's stated revenue impact as additional annual revenue. With investment cost \(I\), annual incremental revenue \(R\), discount rate \(r\), and horizon \(T\), the net present value is

$$
\operatorname{NPV} = -I + \sum_{t=1}^{T} \frac{R}{(1+r)^t}.
$$

The payback period is the first year in which cumulative undiscounted incremental revenue covers \(I\), with linear interpolation within that year. If the investment is not recovered within the modeled horizon, the payback period is reported as unavailable. These scenario assumptions are illustrative; the calculation is not a project appraisal based on observed port accounts.

### What to examine

Inspect the allocation eligibility checks, the simulation timeline, KPI definitions, and the scenario assumptions. Interpret reported metrics as outputs of this synthetic model, not as measured system performance or evidence that a real deployment achieves a specified accuracy, speed, or return.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class VesselStatus(Enum):
    """Operational state used to track a vessel through the simulation."""

    APPROACHING = "approaching"
    AT_ANCHOR = "at_anchor"
    BERTHING = "berthing"
    BERTHED = "berthed"
    DEPARTING = "departing"
    DEPARTED = "departed"


@dataclass
class Vessel:
    """Vessel attributes used by allocation and event-processing rules."""

    id: str
    name: str
    length: float
    draft: float
    cargo_capacity: float
    current_status: VesselStatus
    eta: datetime
    current_position: Tuple[float, float]
    assigned_berth: Optional[int] = None
    cargo_type: str = "General"


@dataclass
class Berth:
    """Berth capacity and occupancy state in the simulated port."""

    id: int
    length: float
    depth: float
    current_vessel: Optional[Vessel] = None
    utilization_rate: float = 0.0
    maintenance_status: str = "Operational"
    structural_health: float = 1.0


@dataclass
class SensorData:
    """A timestamped illustrative sensor observation."""

    sensor_id: str
    sensor_type: str
    timestamp: datetime
    value: float
    unit: str
    location: str


@dataclass
class WeatherData:
    """Environmental conditions used by the monitoring example."""

    timestamp: datetime
    wind_speed: float
    wind_direction: float
    wave_height: float
    visibility: float
    temperature: float


@dataclass
class PredictionResult:
    """A prediction record with a target time and illustrative confidence."""

    prediction_type: str
    target_time: datetime
    predicted_value: float
    confidence: float
    model_version: str

In [ ]:
class DigitalTwinArchitecture:
    """Five-layer data structure for the illustrative port digital twin."""

    def __init__(self, quay_length: float, num_berths: int):
        self.quay_length = quay_length
        self.num_berths = num_berths
        self.berths = self._initialize_berths()
        self.sensors = self._initialize_sensors()
        self.vessels: List[Vessel] = []
        self.ais_data: List[Dict[str, Any]] = []
        self.port_management_data: Dict[str, Any] = {}
        self.predictions: List[PredictionResult] = []
        self.ml_models = self._initialize_ml_models()
        self.simulation_state = "running"
        self.simulation_time = datetime.now()
        self.optimization_engine = OptimizationEngine()
        self.recommendations: List[str] = []
        self.kpi_monitor = KPIMonitor()

    def _initialize_berths(self) -> List[Berth]:
        """Create the eight illustrative berths whose lengths total 3,200 m."""
        berth_configs = [
            (400, 16), (380, 15), (420, 17), (390, 16),
            (410, 16), (400, 16), (390, 15), (410, 17),
        ]
        if len(berth_configs) != self.num_berths:
            raise ValueError("The supplied berth configuration must match num_berths.")
        if not np.isclose(sum(length for length, _ in berth_configs), self.quay_length):
            raise ValueError("The berth lengths must sum to the stated quay length.")
        return [
            Berth(
                id=index + 1,
                length=length,
                depth=depth,
                structural_health=random.uniform(0.85, 1.0),
            )
            for index, (length, depth) in enumerate(berth_configs)
        ]

    def _initialize_sensors(self) -> Dict[str, List[SensorData]]:
        """Create initial synthetic readings for the stated sensor inventory."""
        sensors = {
            "structural_health": [],
            "weather": [],
            "water_level": [],
            "vessel_detection": [],
            "cargo_handling": [],
            "yard_operations": [],
            "environmental": [],
        }
        current_time = datetime.now()

        sensor_groups = [
            ("structural_health", 240, "SH", (0.85, 1.0), "health_index", "Berth"),
            ("weather", 15, "WS", (5.0, 25.0), "knots", "Station"),
            ("water_level", 32, "WL", (12.5, 13.5), "meters", "Monitor"),
            ("vessel_detection", 18, "VR", (0.0, 10.0), "count", "Radar"),
        ]
        for sensor_type, count, prefix, value_range, unit, location_prefix in sensor_groups:
            for index in range(count):
                if sensor_type == "structural_health":
                    location = f"Berth_{(index % 8) + 1}_Section_{(index % 30) + 1}"
                else:
                    location = f"{location_prefix}_{index + 1}"
                sensors[sensor_type].append(
                    SensorData(
                        sensor_id=f"{prefix}_{index:03d}",
                        sensor_type=sensor_type,
                        timestamp=current_time,
                        value=random.uniform(*value_range),
                        unit=unit,
                        location=location,
                    )
                )
        return sensors

    def _initialize_ml_models(self) -> Dict[str, Dict[str, Any]]:
        """Store illustrative model metadata; no models are trained here."""
        now = datetime.now()
        return {
            "vessel_arrival_prediction": {
                "model_type": "LSTM",
                "accuracy": 0.94,
                "prediction_horizon": "48_hours",
                "last_training": now - timedelta(days=7),
            },
            "cargo_volume_forecasting": {
                "model_type": "Random_Forest",
                "accuracy": 0.87,
                "prediction_horizon": "7_days",
                "last_training": now - timedelta(days=3),
            },
            "equipment_failure_prediction": {
                "model_type": "XGBoost",
                "accuracy": 0.91,
                "prediction_horizon": "72_hours",
                "last_training": now - timedelta(days=1),
            },
        }


class OptimizationEngine:
    """Greedy berth allocator subject to length, depth, occupancy, and status."""

    def __init__(self):
        self.constraint_categories = 15
        self.optimization_time_limit = 45
        self.performance_metrics = {
            "avg_decision_time": 0.0,
            "solution_quality": 0.0,
            "constraint_satisfaction": 0.0,
        }

    def optimize_berth_allocation(
        self,
        vessels: List[Vessel],
        berths: List[Berth],
        constraints: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Assign each eligible vessel to the least-utilized suitable berth."""
        del constraints  # The example's explicit eligibility checks define its constraints.
        start_time = time.perf_counter()
        allocation: Dict[str, int] = {}

        for vessel in vessels:
            if vessel.current_status not in (VesselStatus.APPROACHING, VesselStatus.AT_ANCHOR):
                continue
            suitable_berths = [
                berth for berth in berths
                if berth.length >= vessel.length
                and berth.depth >= vessel.draft
                and berth.current_vessel is None
                and berth.maintenance_status == "Operational"
            ]
            if suitable_berths:
                best_berth = min(suitable_berths, key=lambda berth: berth.utilization_rate)
                allocation[vessel.id] = best_berth.id
                best_berth.current_vessel = vessel
                best_berth.utilization_rate = min(1.0, best_berth.utilization_rate + 0.1)

        decision_time = time.perf_counter() - start_time
        return {
            "allocation": allocation,
            "decision_time": decision_time,
            "vessels_assigned": len(allocation),
            "optimization_score": len(allocation) / max(len(vessels), 1),
        }


class KPIMonitor:
    """Maintain current KPIs and a rolling 24-hour history."""

    def __init__(self):
        self.kpis = {
            "berth_utilization": 0.0,
            "vessel_waiting_time": 0.0,
            "cargo_throughput": 0.0,
            "equipment_efficiency": 0.0,
            "safety_incidents": 0,
            "environmental_compliance": 1.0,
            "infrastructure_roi": 0.0,
        }
        self.historical_data: List[Dict[str, Any]] = []

    def update_kpis(
        self,
        berth_data: Dict[str, Any],
        vessel_data: Dict[str, Any],
        cargo_data: Dict[str, Any],
        timestamp: datetime,
    ) -> None:
        """Update occupancy and anchorage-wait metrics from the supplied state."""
        del cargo_data  # Cargo observations are outside this simplified simulation.
        berths = berth_data.get("berths", [])
        occupied = sum(berth.current_vessel is not None for berth in berths)
        self.kpis["berth_utilization"] = occupied / max(len(berths), 1)

        waiting_vessels = [
            vessel for vessel in vessel_data.get("vessels", [])
            if vessel.current_status == VesselStatus.AT_ANCHOR
        ]
        if waiting_vessels:
            waiting_hours = [
                max(0.0, (timestamp - vessel.eta).total_seconds() / 3600)
                for vessel in waiting_vessels
            ]
            self.kpis["vessel_waiting_time"] = float(np.mean(waiting_hours))
        else:
            self.kpis["vessel_waiting_time"] = 0.0

        self.historical_data.append({"timestamp": timestamp, "kpis": self.kpis.copy()})
        cutoff_time = timestamp - timedelta(hours=24)
        self.historical_data = [
            record for record in self.historical_data
            if record["timestamp"] > cutoff_time
        ]

    def get_performance_summary(self) -> Dict[str, Any]:
        """Return current KPIs, recent trends, and threshold-based alerts."""
        return {
            "current_kpis": self.kpis.copy(),
            "trend_analysis": self._calculate_trends(),
            "alerts": self._generate_alerts(),
        }

    def _calculate_trends(self) -> Dict[str, float]:
        """Estimate recent per-observation change for each numeric KPI."""
        if len(self.historical_data) < 2:
            return {}
        recent_data = self.historical_data[-10:]
        trends = {}
        for kpi_name in self.kpis:
            values = [record["kpis"][kpi_name] for record in recent_data]
            if len(values) > 1 and all(isinstance(value, (int, float)) for value in values):
                trends[kpi_name] = (values[-1] - values[0]) / (len(values) - 1)
        return trends

    def _generate_alerts(self) -> List[str]:
        """Apply simple illustrative thresholds to current KPI values."""
        alerts = []
        if self.kpis["berth_utilization"] > 0.9:
            alerts.append("High berth utilization; review capacity and operating plans.")
        if self.kpis["vessel_waiting_time"] > 6.0:
            alerts.append("Vessel waiting time exceeds the illustrative threshold.")
        if self.kpis["environmental_compliance"] < 0.8:
            alerts.append("Environmental compliance is below the illustrative threshold.")
        return alerts

In [ ]:
class VesselTrafficManagementSystem:
    """Illustrative interface for vessel traffic and AIS data."""

    def __init__(self):
        self.ais_range = 50
        self.vessels_tracked: List[Vessel] = []
        self.prediction_accuracy = 0.94

    def process_ais_data(self, raw_data: List[Dict[str, Any]]) -> List[Vessel]:
        """Convert AIS-like records into vessel objects."""
        vessels = []
        for data in raw_data:
            vessel = Vessel(
                id=str(data["mmsi"]),
                name=data.get("name", "Unknown"),
                length=float(data.get("length", 200)),
                draft=float(data.get("draft", 10)),
                cargo_capacity=float(data.get("cargo_capacity", 1000)),
                current_status=VesselStatus.APPROACHING,
                eta=data.get("eta", datetime.now() + timedelta(hours=6)),
                current_position=(float(data["lat"]), float(data["lon"])),
                cargo_type=data.get("cargo_type", "General"),
            )
            vessels.append(vessel)
        self.vessels_tracked = vessels
        return vessels

    def predict_vessel_arrivals(self, time_horizon: int) -> List[PredictionResult]:
        """Generate illustrative hourly arrival counts using a Poisson model."""
        predictions = []
        current_time = datetime.now()
        for hour in range(1, time_horizon + 1):
            predictions.append(
                PredictionResult(
                    prediction_type="vessel_arrival",
                    target_time=current_time + timedelta(hours=hour),
                    predicted_value=float(np.random.poisson(2.5)),
                    confidence=0.94 * (1 - 0.1 * hour / max(time_horizon, 1)),
                    model_version="illustrative_LSTM_metadata",
                )
            )
        return predictions


class CargoHandlingEquipmentSystem:
    """Illustrative crane coordination interface for assigned vessels."""

    def __init__(self):
        self.cranes = 18
        self.equipment_status: Dict[str, str] = {}
        self.coordination_efficiency = 0.87

    def coordinate_with_berths(self, berth_assignments: Dict[str, int]) -> Dict[str, Any]:
        """Assign available cranes and illustrative handling estimates."""
        coordination_plan = {}
        for vessel_id, berth_id in berth_assignments.items():
            required_cranes = random.randint(2, 4)
            available_cranes = [
                f"QC_{index}"
                for index in range(1, self.cranes + 1)
                if f"QC_{index}" not in self.equipment_status
            ]
            if len(available_cranes) >= required_cranes:
                assigned_cranes = available_cranes[:required_cranes]
                coordination_plan[vessel_id] = {
                    "berth_id": berth_id,
                    "assigned_cranes": assigned_cranes,
                    "estimated_handling_time_hours": random.uniform(8, 24),
                    "efficiency_score": random.uniform(0.8, 0.95),
                }
                for crane in assigned_cranes:
                    self.equipment_status[crane] = vessel_id
        return coordination_plan


class YardOperationsSystem:
    """Illustrative interface for yard and intermodal coordination."""

    def __init__(self):
        self.yard_capacity = 2400
        self.truck_connections = 150
        self.rail_connections = 12
        self.utilization_rate = 0.75

    def synchronize_with_berths(self, berth_operations: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Estimate truck and rail allocations from supplied cargo volumes."""
        plan = {
            "truck_allocation": {},
            "rail_allocation": {},
            "yard_utilization": self.utilization_rate,
            "bottleneck_analysis": [],
        }
        for operation in berth_operations:
            vessel_id = operation.get("vessel_id", "Unknown")
            cargo_volume = max(0, operation.get("cargo_volume", 1000))
            plan["truck_allocation"][vessel_id] = min(
                self.truck_connections, int(cargo_volume / 50)
            )
            plan["rail_allocation"][vessel_id] = min(
                self.rail_connections, max(1, int(cargo_volume / 200))
            )
        if self.utilization_rate > 0.85:
            plan["bottleneck_analysis"].append("High yard utilization")
        if self.truck_connections < 100:
            plan["bottleneck_analysis"].append("Limited truck capacity")
        return plan


class EnvironmentalMonitoringSystem:
    """Illustrative weather monitoring and operational-impact rules."""

    def __init__(self):
        self.weather_stations = 15
        self.current_conditions = WeatherData(
            timestamp=datetime.now(),
            wind_speed=12.5,
            wind_direction=225,
            wave_height=1.2,
            visibility=10.0,
            temperature=18.5,
        )

    def assess_operational_impact(self) -> Dict[str, Any]:
        """Apply threshold rules to the current illustrative weather record."""
        assessment = {
            "berthing_restrictions": [],
            "safety_alerts": [],
            "efficiency_impact": 0.0,
            "recommendations": [],
        }
        if self.current_conditions.wind_speed > 25:
            assessment["berthing_restrictions"].append("High wind: limited berthing.")
            assessment["efficiency_impact"] += 0.2
        if self.current_conditions.visibility < 1.0:
            assessment["safety_alerts"].append("Low visibility: apply additional safety protocols.")
            assessment["efficiency_impact"] += 0.15
        if self.current_conditions.wave_height > 2.0:
            assessment["berthing_restrictions"].append("High waves: review berthing conditions.")
            assessment["efficiency_impact"] += 0.25
        if assessment["efficiency_impact"] > 0.1:
            assessment["recommendations"].append("Consider delaying sensitive operations.")
        return assessment

In [ ]:
class PredictiveAnalyticsEngine:
    """Synthetic prediction generators with illustrative model metadata."""

    def __init__(self):
        now = datetime.now()
        self.models = {
            "vessel_arrival_lstm": {
                "accuracy": 0.94,
                "horizon_hours": 48,
                "last_updated": now - timedelta(days=7),
            },
            "cargo_volume_rf": {
                "accuracy": 0.87,
                "horizon_days": 7,
                "last_updated": now - timedelta(days=3),
            },
            "equipment_failure_xgb": {
                "accuracy": 0.91,
                "horizon_hours": 72,
                "last_updated": now - timedelta(days=1),
            },
        }
        self.prediction_cache: List[PredictionResult] = []

    def predict_vessel_arrivals(
        self, historical_data: List[Dict[str, Any]], prediction_horizon: int
    ) -> List[PredictionResult]:
        """Generate synthetic hourly arrival counts for up to 48 hours."""
        del historical_data  # No historical training data is supplied in this example.
        predictions = []
        current_time = datetime.now()
        for hour in range(1, min(max(prediction_horizon, 0), 48) + 1):
            seasonal_factor = 1.0 + 0.3 * np.sin(2 * np.pi * hour / 24)
            predictions.append(
                PredictionResult(
                    prediction_type="vessel_arrival",
                    target_time=current_time + timedelta(hours=hour),
                    predicted_value=float(np.random.poisson(2.5 * seasonal_factor)),
                    confidence=0.94 * (1 - 0.3 * hour / 48),
                    model_version="illustrative_LSTM_metadata",
                )
            )
        self.prediction_cache.extend(predictions)
        return predictions

    def predict_cargo_volumes(
        self, historical_data: List[Dict[str, Any]], prediction_horizon: int
    ) -> List[PredictionResult]:
        """Generate synthetic daily cargo-volume estimates for up to seven days."""
        del historical_data
        predictions = []
        current_time = datetime.now()
        for day in range(1, min(max(prediction_horizon, 0), 7) + 1):
            weekly_factor = 1.0 + 0.2 * np.sin(2 * np.pi * day / 7)
            volume = max(0.0, float(np.random.normal(15000 * weekly_factor, 2000)))
            predictions.append(
                PredictionResult(
                    prediction_type="cargo_volume",
                    target_time=current_time + timedelta(days=day),
                    predicted_value=volume,
                    confidence=0.87 * (1 - 0.2 * day / 7),
                    model_version="illustrative_RF_metadata",
                )
            )
        self.prediction_cache.extend(predictions)
        return predictions

    def predict_equipment_failures(
        self, sensor_data: List[SensorData], prediction_horizon: int
    ) -> List[PredictionResult]:
        """Generate illustrative failure-risk estimates from health readings."""
        predictions = []
        current_time = datetime.now()
        structural_sensors = [
            sensor for sensor in sensor_data
            if sensor.sensor_type == "structural_health"
        ]
        if structural_sensors:
            avg_health = float(np.mean([sensor.value for sensor in structural_sensors[-20:]]))
            base_probability = max(0.0, (0.9 - avg_health) * 0.1)
        else:
            base_probability = 0.0
        for hour in range(1, min(max(prediction_horizon, 0), 72) + 1):
            probability = float(np.clip(base_probability + np.random.normal(0, 0.01), 0, 1))
            predictions.append(
                PredictionResult(
                    prediction_type="equipment_failure",
                    target_time=current_time + timedelta(hours=hour),
                    predicted_value=probability,
                    confidence=0.91 * (1 - 0.3 * hour / 72),
                    model_version="illustrative_XGBoost_metadata",
                )
            )
        self.prediction_cache.extend(predictions)
        return predictions

    def get_model_performance_summary(self) -> Dict[str, Any]:
        """Report configured metadata, not measured predictive performance."""
        return {
            "total_predictions": len(self.prediction_cache),
            "model_accuracy_assumptions": {
                name: model["accuracy"] for name, model in self.models.items()
            },
            "last_training_updates": {
                name: model["last_updated"] for name, model in self.models.items()
            },
            "prediction_horizons": {
                name: model.get("horizon_hours", model.get("horizon_days"))
                for name, model in self.models.items()
            },
        }

In [ ]:
class DigitalTwinSimulation:
    """Coordinate the port state, integrations, analytics, and KPI updates."""

    def __init__(self):
        self.architecture = DigitalTwinArchitecture(quay_length=3200, num_berths=8)
        self.vessel_traffic = VesselTrafficManagementSystem()
        self.cargo_handling = CargoHandlingEquipmentSystem()
        self.yard_operations = YardOperationsSystem()
        self.environmental = EnvironmentalMonitoringSystem()
        self.analytics = PredictiveAnalyticsEngine()
        self.data_update_interval = 30
        self.current_time = datetime.now()
        self.performance_metrics = {
            "data_points_processed": 0,
            "optimization_decisions": 0,
            "prediction_accuracy": 0.0,
            "system_uptime": 0.0,
            "response_time": 0.0,
        }

    def run_simulation(self, duration_hours: int) -> Dict[str, Any]:
        """Advance simulated events in 30-second steps and collect results."""
        if duration_hours < 0:
            raise ValueError("duration_hours must be non-negative.")
        print(f"Digital-twin simulation duration: {duration_hours} hours")
        print("Synthetic data-generation rate assumption: 50,000 points per minute")
        print("External port-system integrations are represented by illustrative interfaces.")

        results = {
            "timeline": [],
            "kpi_evolution": [],
            "optimization_decisions": [],
            "predictions": [],
            "alerts": [],
            "performance_metrics": {},
            "summary": {},
        }
        start_time = time.perf_counter()
        end_time = self.current_time + timedelta(hours=duration_hours)
        vessels = self._generate_vessel_traffic(duration_hours)
        self.architecture.vessels = vessels
        step = 0
        step_seconds = self.data_update_interval

        while self.current_time < end_time:
            self._update_sensor_data()
            self.performance_metrics["data_points_processed"] += 25000
            self._process_vessel_events(vessels)

            elapsed_simulation_seconds = step * step_seconds
            if elapsed_simulation_seconds % 300 == 0:
                optimization_result = self._run_optimization()
                results["optimization_decisions"].append(
                    {"timestamp": self.current_time, "result": optimization_result}
                )
                self.performance_metrics["optimization_decisions"] += 1

            if elapsed_simulation_seconds % 3600 == 0:
                results["predictions"].append(
                    {"timestamp": self.current_time, "predictions": self._update_predictions()}
                )

            self._update_kpis()
            alerts = self.architecture.kpi_monitor._generate_alerts()
            if alerts:
                results["alerts"].append(
                    {"timestamp": self.current_time, "alerts": alerts}
                )

            if elapsed_simulation_seconds % 1800 == 0:
                timeline_record = {
                    "timestamp": self.current_time,
                    "berth_utilization": self.architecture.kpi_monitor.kpis["berth_utilization"],
                    "vessel_waiting_time": self.architecture.kpi_monitor.kpis["vessel_waiting_time"],
                    "active_vessels": sum(
                        vessel.current_status != VesselStatus.DEPARTED for vessel in vessels
                    ),
                }
                results["timeline"].append(timeline_record)
                results["kpi_evolution"].append(
                    self.architecture.kpi_monitor.kpis.copy()
                )

            self.current_time += timedelta(seconds=step_seconds)
            step += 1

        total_wall_time = time.perf_counter() - start_time
        self.performance_metrics["system_uptime"] = 100.0
        simulated_seconds = duration_hours * 3600
        self.performance_metrics["response_time"] = (
            total_wall_time / simulated_seconds if simulated_seconds else 0.0
        )
        results["performance_metrics"] = self.performance_metrics.copy()
        results["summary"] = self._generate_simulation_summary(results)
        return results

    def _generate_vessel_traffic(self, duration_hours: int) -> List[Vessel]:
        """Generate synthetic hourly arrivals and vessel characteristics."""
        vessels = []
        vessel_types = [
            ("Feeder", 180, 10, 800),
            ("Panamax", 250, 12, 2000),
            ("Post-Panamax", 320, 14, 4000),
            ("ULCV", 400, 16, 8000),
        ]
        vessel_id = 1
        for hour in range(duration_hours):
            for _ in range(np.random.poisson(2.5)):
                vessel_type = random.choice(vessel_types)
                vessel = Vessel(
                    id=f"V_{vessel_id:04d}",
                    name=f"Vessel {vessel_id}",
                    length=vessel_type[1],
                    draft=vessel_type[2],
                    cargo_capacity=vessel_type[3],
                    current_status=VesselStatus.APPROACHING,
                    eta=self.current_time + timedelta(
                        hours=hour + random.uniform(2, 8)
                    ),
                    current_position=(
                        random.uniform(40.5, 41.0),
                        random.uniform(-71.0, -70.5),
                    ),
                    cargo_type=vessel_type[0],
                )
                vessels.append(vessel)
                vessel_id += 1
        return vessels

    def _update_sensor_data(self) -> None:
        """Advance synthetic sensor readings at the configured update interval."""
        current_time = self.current_time
        for sensor in self.architecture.sensors["structural_health"]:
            sensor.timestamp = current_time
            sensor.value = float(np.clip(sensor.value + np.random.normal(0, 0.01), 0.8, 1.0))
        for sensor in self.architecture.sensors["weather"]:
            sensor.timestamp = current_time
            sensor.value = random.uniform(5, 30)
        for sensor in self.architecture.sensors["water_level"]:
            sensor.timestamp = current_time
            tidal_factor = 0.5 * np.sin(
                2 * np.pi * current_time.hour / 12.42
            )
            sensor.value = 13.0 + tidal_factor + np.random.normal(0, 0.1)

    def _process_vessel_events(self, vessels: List[Vessel]) -> None:
        """Move arrivals to anchorage and release berths after scheduled departures."""
        for vessel in vessels:
            if (
                vessel.current_status == VesselStatus.APPROACHING
                and vessel.eta <= self.current_time
            ):
                vessel.current_status = VesselStatus.AT_ANCHOR

            if (
                vessel.current_status == VesselStatus.BERTHED
                and vessel.eta <= self.current_time
            ):
                if vessel.assigned_berth is not None:
                    berth = next(
                        berth for berth in self.architecture.berths
                        if berth.id == vessel.assigned_berth
                    )
                    berth.current_vessel = None
                    berth.utilization_rate = max(0.0, berth.utilization_rate - 0.1)
                vessel.current_status = VesselStatus.DEPARTED

    def _run_optimization(self) -> Dict[str, Any]:
        """Allocate waiting vessels, then schedule their illustrative departures."""
        waiting_vessels = [
            vessel for vessel in self.architecture.vessels
            if vessel.current_status == VesselStatus.AT_ANCHOR
        ]
        result = self.architecture.optimization_engine.optimize_berth_allocation(
            waiting_vessels, self.architecture.berths, {}
        )
        for vessel in waiting_vessels:
            if vessel.id in result["allocation"]:
                vessel.assigned_berth = result["allocation"][vessel.id]
                vessel.current_status = VesselStatus.BERTHED
                vessel.eta = self.current_time + timedelta(hours=random.uniform(8, 24))
        return {
            "decision_time": result["decision_time"],
            "vessels_processed": len(waiting_vessels),
            "vessels_assigned": result["vessels_assigned"],
            "optimization_score": result["optimization_score"],
            "within_time_limit": result["decision_time"] < 45,
        }

    def _update_predictions(self) -> Dict[str, Any]:
        """Generate prediction records and summarize their configured confidence."""
        historical_data: List[Dict[str, Any]] = []
        arrival_predictions = self.analytics.predict_vessel_arrivals(historical_data, 48)
        cargo_predictions = self.analytics.predict_cargo_volumes(historical_data, 7)
        failure_predictions = self.analytics.predict_equipment_failures(
            self.architecture.sensors["structural_health"], 72
        )
        all_predictions = arrival_predictions + cargo_predictions + failure_predictions
        return {
            "arrival_predictions": len(arrival_predictions),
            "cargo_predictions": len(cargo_predictions),
            "failure_predictions": len(failure_predictions),
            "average_configured_confidence": float(
                np.mean([prediction.confidence for prediction in all_predictions])
            ) if all_predictions else 0.0,
        }

    def _update_kpis(self) -> None:
        """Pass the current simulated state to the KPI monitor."""
        self.architecture.kpi_monitor.update_kpis(
            {"berths": self.architecture.berths},
            {"vessels": self.architecture.vessels},
            {},
            self.current_time,
        )

    def _generate_simulation_summary(self, results: Dict[str, Any]) -> Dict[str, Any]:
        """Summarize recorded KPIs and allocation decisions without adding outcomes."""
        timeline = results["timeline"]
        decisions = results["optimization_decisions"]
        avg_utilization = float(np.mean([row["berth_utilization"] for row in timeline])) if timeline else 0.0
        avg_waiting = float(np.mean([row["vessel_waiting_time"] for row in timeline])) if timeline else 0.0
        avg_decision_time = float(
            np.mean([row["result"]["decision_time"] for row in decisions])
        ) if decisions else 0.0
        return {
            "simulation_duration_hours": len(timeline) / 2,
            "average_berth_utilization": avg_utilization,
            "average_waiting_time_hours": avg_waiting,
            "total_optimization_decisions": len(decisions),
            "average_decision_time_seconds": avg_decision_time,
            "total_vessels_assigned": sum(
                row["result"]["vessels_assigned"] for row in decisions
            ),
            "total_alert_records": len(results["alerts"]),
            "system_performance": self.performance_metrics.copy(),
        }

In [ ]:
def run_digital_twin_simulation() -> Tuple[DigitalTwinSimulation, Dict[str, Any]]:
    """Initialize the illustrative port and run a seven-day simulation."""
    print("Initializing the illustrative Mediterranean Logistics Hub digital twin")
    print("Quay length: 3.2 km across 8 berths")
    print("Structural-health sensors: 240 | Weather stations: 15")
    print("Water-level monitors: 32 | Vessel-detection radars: 18")
    print("Cargo-handling cranes: 18 | Illustrative yard capacity: 2,400 TEU")
    print("Data-rate assumption: 50,000 synthetic points per minute")
    digital_twin = DigitalTwinSimulation()
    simulation_results = digital_twin.run_simulation(duration_hours=168)
    return digital_twin, simulation_results


digital_twin, simulation_results = run_digital_twin_simulation()
simulation_summary = simulation_results["summary"]
print("\nSimulation summary (synthetic model outputs):")
for metric, value in simulation_summary.items():
    print(f"{metric}: {value}")

In [ ]:
def visualize_digital_twin_results(
    digital_twin: DigitalTwinSimulation,
    results: Dict[str, Any],
) -> None:
    """Plot recorded KPIs, allocation timing, and model-scope indicators."""
    del digital_twin  # The plotted values are supplied explicitly in results.
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    timeline = results["timeline"]
    summary = results["summary"]

    if timeline:
        timestamps = [row["timestamp"] for row in timeline]
        utilization = [row["berth_utilization"] * 100 for row in timeline]
        active_vessels = [row["active_vessels"] for row in timeline]
        line1 = ax1.plot(timestamps, utilization, color="tab:blue", label="Berth utilization")
        ax1_twin = ax1.twinx()
        line2 = ax1_twin.plot(timestamps, active_vessels, color="tab:green", label="Active vessels")
        ax1.set_xlabel("Simulation time")
        ax1.set_ylabel("Berth utilization (%)")
        ax1_twin.set_ylabel("Active vessels (count)")
        ax1.set_title("Recorded operational indicators")
        ax1.grid(True, alpha=0.3)
        lines = line1 + line2
        ax1.legend(lines, [line.get_label() for line in lines], loc="upper left")
    else:
        ax1.text(0.5, 0.5, "No timeline records", ha="center", va="center")
        ax1.set_title("Recorded operational indicators")

    metric_names = ["Average berth utilization", "Average waiting time", "Decision time"]
    metric_values = [
        summary["average_berth_utilization"] * 100,
        summary["average_waiting_time_hours"],
        summary["average_decision_time_seconds"],
    ]
    bars = ax2.bar(metric_names, metric_values, color=["#2E86AB", "#A23B72", "#F18F01"])
    ax2.set_ylabel("Value (mixed units; see labels)")
    ax2.set_title("Recorded summary metrics")
    ax2.tick_params(axis="x", rotation=20)
    ax2.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, metric_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    decisions = results["optimization_decisions"]
    if decisions:
        decision_times = [row["result"]["decision_time"] * 1000 for row in decisions]
        assigned = [row["result"]["vessels_assigned"] for row in decisions]
        ax3.scatter(decision_times, assigned, alpha=0.6, color="#F18F01", label="Simulation decisions")
        ax3.axvline(45000, color="red", linestyle="--", label="45-second reference limit")
        ax3.set_xlabel("Decision time (milliseconds)")
        ax3.set_ylabel("Vessels assigned per decision (count)")
        ax3.set_title("Greedy allocation timing")
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    else:
        ax3.text(0.5, 0.5, "No allocation decisions", ha="center", va="center")
        ax3.set_title("Greedy allocation timing")

    ax4.axis("off")
    status_text = (
        "MODEL SCOPE\n"
        "Synthetic time-stepped simulation\n"
        "Greedy berth-allocation rule\n"
        "Prediction records are generated synthetically\n"
        "Accuracy figures are assumptions, not validation results\n\n"
        f"Recorded allocation decisions: {summary['total_optimization_decisions']}\n"
        f"Recorded vessel assignments: {summary['total_vessels_assigned']}\n"
        f"Recorded alert events: {summary['total_alert_records']}\n"
        f"Average berth utilization: {summary['average_berth_utilization']:.1%}"
    )
    ax4.text(
        0.05,
        0.95,
        status_text,
        transform=ax4.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightgray", "alpha": 0.8},
    )
    fig.suptitle("Illustrative Digital-Twin Simulation", fontsize=16, fontweight="bold")
    fig.tight_layout()
    plt.show()


visualize_digital_twin_results(digital_twin, simulation_results)

In [ ]:
def simulate_scenario(
    digital_twin: DigitalTwinSimulation,
    scenario_config: Dict[str, Any],
) -> Dict[str, Any]:
    """Evaluate stated incremental-revenue assumptions for one scenario."""
    base_utilization = digital_twin.architecture.kpi_monitor.kpis.get(
        "berth_utilization", 0.75
    )
    base_waiting_time = digital_twin.architecture.kpi_monitor.kpis.get(
        "vessel_waiting_time", 4.0
    )
    modifications = scenario_config.get("modifications", {})
    utilization_impact = 0.0
    waiting_time_impact = 0.0
    annual_incremental_revenue = 0.0
    investment_cost = 0.0

    if "berth_6_length" in modifications:
        utilization_impact += 0.05
        waiting_time_impact -= 0.10
        annual_incremental_revenue += 2_000_000
        investment_cost += 8_500_000
    if "num_berths" in modifications:
        utilization_impact += 0.15
        waiting_time_impact -= 0.25
        annual_incremental_revenue += 5_000_000
        investment_cost += 15_000_000
    if "depth_increase" in modifications:
        utilization_impact += 0.08
        waiting_time_impact -= 0.05
        annual_incremental_revenue += 3_000_000
        investment_cost += 8_000_000

    new_utilization = min(0.95, base_utilization + utilization_impact)
    new_waiting_time = max(0.5, base_waiting_time * (1 + waiting_time_impact))
    discount_rate = 0.08
    years = 15
    npv = -investment_cost + sum(
        annual_incremental_revenue / ((1 + discount_rate) ** year)
        for year in range(1, years + 1)
    )

    payback_period = None
    if investment_cost == 0:
        payback_period = 0.0
    elif annual_incremental_revenue > 0:
        undiscounted_payback = investment_cost / annual_incremental_revenue
        if undiscounted_payback <= years:
            payback_period = undiscounted_payback

    return {
        "berth_utilization": new_utilization,
        "waiting_time": new_waiting_time,
        "annual_incremental_revenue": annual_incremental_revenue,
        "npv": npv,
        "payback_period": payback_period,
        "investment_cost": investment_cost,
    }


def what_if_analysis(digital_twin: DigitalTwinSimulation) -> Dict[str, Dict[str, Any]]:
    """Compare the baseline with four illustrative infrastructure scenarios."""
    scenarios = {
        "Baseline": {
            "description": "Current configuration: 8 berths and 3.2 km of quay.",
            "modifications": {},
        },
        "Berth_6_Extension": {
            "description": "Extend Berth 6 by 85 m, from 410 m to 495 m.",
            "modifications": {"berth_6_length": 495},
        },
        "Additional_Berth": {
            "description": "Add a ninth berth with a stated total quay length of 3,600 m.",
            "modifications": {"num_berths": 9, "total_length": 3600},
        },
        "Deepening_Project": {
            "description": "Deepen all berths by 2 m.",
            "modifications": {"depth_increase": 2.0},
        },
        "Combined_Upgrade": {
            "description": "Combine the Berth 6 extension, ninth berth, and deepening.",
            "modifications": {
                "berth_6_length": 495,
                "num_berths": 9,
                "total_length": 3600,
                "depth_increase": 2.0,
            },
        },
    }
    scenario_results = {}
    for name, config in scenarios.items():
        result = simulate_scenario(digital_twin, config)
        scenario_results[name] = result
        payback = (
            "not recovered within the modeled horizon"
            if result["payback_period"] is None
            else f"{result['payback_period']:.2f} years"
        )
        print(f"\n{name}: {config['description']}")
        print(f"  Berth utilization: {result['berth_utilization']:.1%}")
        print(f"  Waiting time: {result['waiting_time']:.2f} hours")
        print(f"  Annual incremental revenue assumption: ${result['annual_incremental_revenue']:,.0f}")
        print(f"  Investment assumption: ${result['investment_cost']:,.0f}")
        print(f"  15-year NPV: ${result['npv']:,.0f}")
        print(f"  Payback period: {payback}")

    names = list(scenario_results)
    colors = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#E71D36"]
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

    utilization = [scenario_results[name]["berth_utilization"] * 100 for name in names]
    ax1.bar(names, utilization, color=colors)
    ax1.set_ylabel("Berth utilization (%)")
    ax1.set_title("Illustrative utilization by scenario")
    ax1.tick_params(axis="x", rotation=25)
    ax1.grid(True, alpha=0.3, axis="y")

    waiting = [scenario_results[name]["waiting_time"] for name in names]
    ax2.bar(names, waiting, color=colors)
    ax2.set_ylabel("Average waiting time (hours)")
    ax2.set_title("Illustrative waiting time by scenario")
    ax2.tick_params(axis="x", rotation=25)
    ax2.grid(True, alpha=0.3, axis="y")

    npvs = [scenario_results[name]["npv"] / 1e6 for name in names]
    ax3.bar(names, npvs, color=colors)
    ax3.set_ylabel("15-year NPV (USD millions)")
    ax3.set_title("NPV under stated financial assumptions")
    ax3.tick_params(axis="x", rotation=25)
    ax3.grid(True, alpha=0.3, axis="y")

    paybacks = [
        np.nan if scenario_results[name]["payback_period"] is None
        else scenario_results[name]["payback_period"]
        for name in names
    ]
    ax4.bar(names, paybacks, color=colors)
    ax4.set_ylabel("Payback period (years)")
    ax4.set_title("Undiscounted payback within 15 years")
    ax4.tick_params(axis="x", rotation=25)
    ax4.grid(True, alpha=0.3, axis="y")

    fig.suptitle("Illustrative Infrastructure Scenario Comparison", fontsize=16)
    fig.tight_layout()
    plt.show()

    best_npv = max(result["npv"] for result in scenario_results.values())
    best_scenarios = [
        name for name, result in scenario_results.items()
        if np.isclose(result["npv"], best_npv, rtol=1e-9, atol=1e-6)
    ]
    print("\nHighest modeled NPV under the stated assumptions:")
    print(", ".join(best_scenarios))
    print("This ranking is conditional on the illustrative scenario assumptions.")
    return scenario_results


what_if_results = what_if_analysis(digital_twin)

## Interpretation, strengths, and limitations

The notebook connects asset records, vessel events, synthetic sensor updates, analytics records, berth allocation, KPI monitoring, and infrastructure scenarios in one simulation workflow. The connection between layers is the central teaching point: a decision-support component can use the current state produced by the physical and operational layers, while the simulation records the consequences for later reporting.

The allocation engine applies the stated greedy rule, not a mathematical-programming solver. It checks berth length, depth, occupancy, and operational status, then chooses the least-utilized eligible berth for each vessel in sequence. Consequently, its reported assignments describe the behavior of that rule on the generated instance; they do not establish that the assignments are globally optimal or that every operational constraint has been represented.

The synthetic prediction records illustrate data structures and time horizons. The accuracy values stored in model metadata are assumptions and are not calculated from validation data. Similarly, the scenario comparisons are conditional on explicitly stated capacity, revenue, investment, discount-rate, and horizon assumptions. A real investment recommendation would require validated demand, engineering estimates, operating costs, and financial data.

### Strengths of this modeling tier

- It makes the interfaces among physical assets, operations, analytics, simulation, and decision support visible.
- It provides a time-dependent setting for studying vessel arrivals, berth occupancy, departures, and KPI monitoring.
- It supports structured what-if comparisons using a consistent set of stated scenario assumptions.
- Its modular classes make assumptions and data flows inspectable.

### Limitations and appropriate use

- Sensor observations, vessel arrivals, prediction records, and financial assumptions are synthetic or illustrative; they are not live operational evidence.
- The model does not implement a validated machine-learning pipeline, real-time data synchronization, production-grade interoperability, or comprehensive port optimization.
- Greedy allocation is order-dependent and can miss better joint assignments. Its role here is to demonstrate the integrated simulation workflow, not to certify a berth plan.
- The scenario model omits important engineering, operational, environmental, and financial details, so its NPV ranking should not be used as an investment decision.

### When this modeling approach is useful

An integrated simulation is useful for organizing subsystem interfaces, exploring how assumptions influence system behavior, and identifying which data would be needed for a more realistic model. Further work can refine the existing subsystem models, add validated constraints and observations, and test sensitivity to assumptions without treating illustrative outputs as measured performance.